# 02 · EDA de las predicciones — Etapa 1 zero-shot sobre M3DI-*test* ($n = 10\,000$)

**Pregunta.** Dado que cada imagen de Multimodal3DIdent se genera a partir de factores conocidos, ¿qué describen realmente los cinco modelos
bajo cada uno de los seis prompts, y dónde fallan: en *percibir* el atributo o en *decirlo* en el formato esperado?

Cada imagen $i$ proviene de
$$
z_i=\big(s_i,\;x_i,\;y_i,\;h_i\big),\qquad s_i\in\mathcal S=\{\text{teapot},\dots,\text{head}\},\;|\mathcal S|=7,\qquad (x_i,y_i)\in\{0,1,2\}^2,\qquad h_i\in[0,1),
$$
donde $h_i$ es el tono del objeto; la referencia nombra además un color $c_i$ de Matplotlib (`tab:`, `xkcd:` o CSS) cercano a $h_i$.
Cada corrida $r=(\text{modelo},\text{prompt})$ produce un texto $\hat t_{i,r}$, del que un extractor léxico —validado sobre las referencias en §4—
obtiene $\hat z_{i,r}$ atributo por atributo.

| § | Contenido |
|---|---|
| 1 | Configuración y parámetros |
| 2 | Carga de `runs/<exp_id>/` e integridad de las corridas |
| 3 | Factores verdaderos y líneas base |
| 4 | Extractor de atributos y su validación sobre las referencias |
| 5 | Texto: longitud, truncamiento, etiquetas `<think>`, JSON, diversidad |
| 6 | Copia de los ejemplos del prompt (p4 vs p4m) |
| 7 | Fidelidad por atributo: mención, exactitud, exactitud condicional |
| 8 | Sensibilidad al prompt y cobertura vs. exactitud |
| 9 | Control Qwen3.5 Base vs Instruct (McNemar exacto + Holm) |
| 10 | Errores con el mejor prompt de cada modelo: forma, posición, tono |
| 11 | Comparaciones pareadas entre modelos y dificultad por imagen |
| 12 | Ejemplos cualitativos |
| 13 | Exportación y limitaciones |

> **VL-JEPA no genera texto**: recupera la descripción más cercana de un banco de captions de *train*. Sus salidas tienen el formato de la
> referencia por construcción, así que sus tasas de mención y de uso de nombres entre comillas no son comparables con las de los modelos
> generativos. La exactitud por atributo sí lo es.

**Cómo ejecutarlo.** Solo lee `data/` y `runs/` (sin GPU). Funciona con corridas incompletas: las marca en §2 y las comparaciones pareadas
usan la intersección de imágenes. En el cluster, en segundo plano:

```bash
cd /mnt/home2/t7-vLLM/Multimodal_LLM && setsid nohup uv run --no-sync jupyter nbconvert --to notebook --execute --inplace \
  --ExecutePreprocessor.timeout=-1 notebooks/02_eda_predicciones.ipynb > logs/eda_n10000.log 2>&1 < /dev/null &
```

## 1. Configuración

In [ ]:
import colorsys
import json
import math
import re
import sys
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import yaml
from IPython.display import Markdown, display

try:
    from vlmfid.paths import MANIFESTS, ROOT, RUNS
except ImportError:                      # kernel sin el paquete instalado en modo editable
    sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))
    from vlmfid.paths import MANIFESTS, ROOT, RUNS
from vlmfid.data import data_root, load_manifest, read_manifest
from vlmfid.eval import copy_flags, prompt_examples
from vlmfid.prompts import parse_json_response, strip_think_tags

# ── Parámetros ───────────────────────────────────────────────────────────────
STAGE, SPLIT, VARIANT = "zeroshot", "test", "base"
LIMIT = 10_000            # tamaño de la corrida; con el test completo (10 000) incluye también runs con data.limit=null
SUBSET_SEED = 0
INCLUDE_PARTIAL = True    # incluir corridas aún en curso (quedan marcadas en §2)
STRICT_JSON = True        # p3_json: si no hay JSON recuperable, los atributos cuentan como omitidos (fallo de formato)
ALPHA = 0.05
SEED = 20260928
TAG = "eda_n10000"
FIG_DIR = ROOT / "reports" / "figures" / TAG
TAB_DIR = ROOT / "reports" / "tables" / TAG
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

MODEL_LABELS = {                       # model.name -> etiqueta; el orden define el de figuras y tablas
    "vljepa_open_msrvtt": "VL-JEPA",
    "llava_ov_05b": "LLaVA-OV 0.5B",
    "qwen35_08b_base": "Qwen3.5 Base",
    "qwen35_08b_instruct": "Qwen3.5 Instruct",
    "internvl3_1b": "InternVL3 1B",
}
RETRIEVAL = {"vljepa_open_msrvtt"}     # modelos que recuperan un caption en vez de generarlo
PROMPT_CATALOG = yaml.safe_load((ROOT / "configs" / "prompts.yaml").read_text(encoding="utf-8"))
PROMPT_SHORT = {p: ("p4m" if p.startswith("p4m") else p.split("_")[0]) for p in PROMPT_CATALOG}
JSON_PROMPTS = {p for p, c in PROMPT_CATALOG.items() if c.get("response_format") == "json"}

RNG = np.random.default_rng(SEED)
Z = 1.959963984540054                  # cuantil 0.975 de la normal estándar (ALPHA = 0.05)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)
warnings.filterwarnings("ignore", category=FutureWarning)
print(f"ROOT = {ROOT}\nRUNS = {RUNS}\nsalidas -> {FIG_DIR.relative_to(ROOT)}, {TAB_DIR.relative_to(ROOT)}")

In [ ]:
# ── Estilo (mismo sistema que las figuras de reports/figures/eval) ────────────
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
INK, INK2, INK3, GRID, SURF, NEUTRAL = "#0b0b0b", "#52514e", "#8a8984", "#e6e5e1", "#fcfcfb", "#c9c8c2"
SEQ = mcolors.LinearSegmentedColormap.from_list(
    "seq_blue", ["#f4f8fe", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
DIV = mcolors.LinearSegmentedColormap.from_list("div_red_blue", ["#c23a39", "#ee9190", "#f0efec", "#6da7ec", "#184f95"])
MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]

plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": INK3, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.titlesize": 10.5, "axes.titleweight": "semibold", "axes.labelsize": 9.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "xtick.color": INK2, "ytick.color": INK2, "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "legend.frameon": False, "legend.fontsize": 8.5, "font.size": 9.5,
    "lines.linewidth": 2, "lines.markersize": 7, "figure.dpi": 110,
})
PCT = plt.FuncFormatter(lambda v, _: f"{v:.0%}")


def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=200, bbox_inches="tight")


def save_table(df, name, index=True, float_format="%.3f"):
    df.to_csv(TAB_DIR / f"{name}.csv", index=index)
    try:
        df.to_latex(TAB_DIR / f"{name}.tex", index=index, float_format=float_format, escape=True, na_rep="--")
    except Exception as e:                       # to_latex requiere jinja2
        print(f"[aviso] {name}.tex no generado: {e}")


def heat(ax, M, xlabels, ylabels, fmt="{:.0%}", vmin=0.0, vmax=1.0, cmap=SEQ, fontsize=8):
    # Mapa de calor anotado; las celdas sin dato quedan en gris con '–'.
    M = np.asarray(M, dtype=float)
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    ax.imshow(np.where(np.isfinite(M), M, np.nan), cmap=cmap, norm=norm, aspect="auto")
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]
            if np.isfinite(v):
                r, g, b, _ = cmap(norm(v))
                lum = 0.2126 * r + 0.7152 * g + 0.0722 * b
                ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=fontsize, color="white" if lum < 0.5 else INK)
            else:
                ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, color=GRID, lw=0))
                ax.text(j, i, "–", ha="center", va="center", fontsize=fontsize, color=INK3)
    ax.set_xticks(range(M.shape[1]), xlabels)
    ax.set_yticks(range(M.shape[0]), ylabels)
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.tick_params(length=0)


def wilson(k, n, z=Z):
    # Intervalo de Wilson para una proporción (vectorizado); NaN si n = 0.
    k, n = np.asarray(k, float), np.asarray(n, float)
    with np.errstate(invalid="ignore", divide="ignore"):
        p = k / n
        den = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / den
        hw = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return c - hw, c + hw


def mcnemar_exact(b, c):
    # p-valor bilateral exacto de McNemar: bajo H0, b ~ Bin(b + c, 1/2).
    n = int(b + c)
    if n == 0:
        return 1.0
    k = int(min(b, c))
    logs = [math.lgamma(n + 1) - math.lgamma(i + 1) - math.lgamma(n - i + 1) - n * math.log(2) for i in range(k + 1)]
    m = max(logs)
    return min(1.0, 2 * math.exp(m + math.log(sum(math.exp(l - m) for l in logs))))


def holm(p):
    # Ajuste de Holm–Bonferroni (controla la FWER).
    p = np.asarray(p, float)
    order, m = np.argsort(p), len(p)
    adj, run = np.empty(m), 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i])
        adj[i] = min(1.0, run)
    return adj

## 2. Carga e integridad

Se recorren `runs/*/config.yaml` y se conservan las corridas con `stage = zeroshot`, `split = test`, `variant = base` y tamaño $n=10\,000$.
Modelo y prompt se leen de la **configuración resuelta** (no del nombre de la carpeta). Las imágenes esperadas son las que devuelve
`load_manifest` con la misma semilla que usó `infer.py`, de modo que todas las corridas se comparan sobre el mismo conjunto.

In [ ]:
man_full = read_manifest(SPLIT, variant=VARIANT)
N_MAN = man_full.height
expected_ids = [r["id"] for r in load_manifest(SPLIT, limit=LIMIT, subset_seed=SUBSET_SEED, variant=VARIANT)]
EXPECTED = set(expected_ids)
N_EXP = len(expected_ids)


def same_size(limit):
    # data.limit=10000 y data.limit=null son el mismo conjunto cuando el split tiene exactamente 10 000 imágenes.
    eff = N_MAN if limit in (None, "null") else min(int(limit), N_MAN)
    return eff == min(LIMIT, N_MAN)


meta_rows, frames, skipped = [], [], Counter()
for cfg_path in sorted(RUNS.glob("*/config.yaml")):
    cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8")) or {}
    data = cfg.get("data") or {}
    if (cfg.get("stage") != STAGE or data.get("split") != SPLIT or data.get("variant", "base") != VARIANT
            or not same_size(data.get("limit"))):
        skipped["otro stage/split/tamaño"] += 1
        continue
    d = cfg_path.parent
    st = json.loads((d / "status.json").read_text()) if (d / "status.json").exists() else {}
    ids, preds, bad = [], [], 0
    pred_path = d / "predictions.jsonl"
    if pred_path.exists():
        with pred_path.open(encoding="utf-8") as f:
            for line in f:
                try:
                    o = json.loads(line)
                except json.JSONDecodeError:
                    bad += 1                         # línea truncada por una caída
                    continue
                ids.append(o["id"])
                preds.append(o.get("prediction") or "")
    model, prompt = cfg["model"]["name"], cfg["prompt"]["id"]
    ser = pd.Series(ids, dtype="object")
    meta_rows.append({
        "exp_id": cfg.get("exp_id", d.name), "model": model, "prompt": prompt,
        "prompt_text": " ".join(str(cfg["prompt"].get("text", "")).split()),
        "estado": st.get("status", "desconocido"), "filas": len(ids), "duplicados": int(ser.duplicated().sum()),
        "fuera_del_conjunto": int((~ser.isin(EXPECTED)).sum()), "lineas_corruptas": bad,
        "img_por_s": st.get("img_per_s"), "pico_gb": st.get("peak_mem_gb"), "intentos": st.get("attempts"),
        "error": st.get("error"),
    })
    frames.append(pd.DataFrame({"exp_id": meta_rows[-1]["exp_id"], "id": ids, "prediction": preds}))

assert meta_rows, f"No hay corridas de {STAGE}/{SPLIT} con n={LIMIT} en {RUNS}"
runs_meta = pd.DataFrame(meta_rows)
runs_meta["faltantes"] = [len(EXPECTED - set(f["id"])) for f in frames]
runs_meta["cobertura"] = 1 - runs_meta["faltantes"] / N_EXP
runs_meta["completa"] = runs_meta["faltantes"].eq(0) & runs_meta["estado"].eq("completed")

# Dos corridas del mismo (modelo, prompt) —p. ej. data.limit=10000 y null— : se conserva la de mayor cobertura.
dups = runs_meta[runs_meta.duplicated(["model", "prompt"], keep=False)]
if len(dups):
    print("[aviso] (modelo, prompt) repetidos; se conserva el de mayor cobertura:")
    display(dups[["exp_id", "filas", "estado"]])
keep = runs_meta.sort_values("cobertura", ascending=False).drop_duplicates(["model", "prompt"]).index
if not INCLUDE_PARTIAL:
    keep = keep.intersection(runs_meta.index[runs_meta["completa"]])
runs_meta = runs_meta.loc[sorted(keep)].copy()
pred = pd.concat(frames, ignore_index=True)
pred = pred[pred["exp_id"].isin(runs_meta["exp_id"]) & pred["id"].isin(EXPECTED)]
pred = pred.drop_duplicates(["exp_id", "id"], keep="last")

# Orden y etiquetas
MODEL_ORDER = [m for m in MODEL_LABELS if m in set(runs_meta["model"])] + \
              sorted(set(runs_meta["model"]) - set(MODEL_LABELS))
PROMPT_ORDER = [p for p in PROMPT_CATALOG if p in set(runs_meta["prompt"])] + \
               sorted(set(runs_meta["prompt"]) - set(PROMPT_CATALOG))
ML = {m: MODEL_LABELS.get(m, m) for m in MODEL_ORDER}
PS = {p: PROMPT_SHORT.get(p, p) for p in PROMPT_ORDER}
MODEL_COLORS = {m: SERIES[i % len(SERIES)] for i, m in enumerate(MODEL_ORDER)}
MODEL_MARKERS = {m: MARKERS[i % len(MARKERS)] for i, m in enumerate(MODEL_ORDER)}
runs_meta["_m"] = runs_meta["model"].map(MODEL_ORDER.index)
runs_meta["_p"] = runs_meta["prompt"].map(PROMPT_ORDER.index)
runs_meta = runs_meta.sort_values(["_m", "_p"]).drop(columns=["_m", "_p"]).reset_index(drop=True)
runs_meta["run"] = runs_meta["model"].map(ML) + " · " + runs_meta["prompt"].map(PS)
RUN_ORDER = runs_meta["run"].tolist()
E2R = dict(zip(runs_meta["exp_id"], runs_meta["run"]))
pred = pred.merge(runs_meta[["exp_id", "model", "prompt", "run"]], on="exp_id")

print(f"Manifest {SPLIT}: {N_MAN:,} imágenes · conjunto esperado: {N_EXP:,} · corridas: {len(runs_meta)} "
      f"({runs_meta['completa'].sum()} completas) · predicciones: {len(pred):,} · omitidas: {dict(skipped)}")
display(pd.DataFrame({"prompt": PROMPT_ORDER, "abreviatura": [PS[p] for p in PROMPT_ORDER]}).set_index("prompt").T)

In [ ]:
cols = ["run", "estado", "filas", "cobertura", "faltantes", "duplicados", "fuera_del_conjunto", "lineas_corruptas",
        "img_por_s", "pico_gb", "intentos"]
integ = runs_meta[cols].set_index("run")
display(integ.style.format({"cobertura": "{:.1%}", "img_por_s": "{:.1f}", "pico_gb": "{:.2f}"}, na_rep="–")
        .apply(lambda s: ["color: #c23a39; font-weight: 600" if v < 1 else "" for v in s], subset=["cobertura"]))
save_table(integ, "integridad_corridas")
if runs_meta["error"].notna().any():
    display(runs_meta.loc[runs_meta["error"].notna(), ["run", "estado", "error"]])
incompletas = runs_meta.loc[~runs_meta["completa"], "run"].tolist()
if incompletas:
    display(Markdown(f"**Corridas incompletas ({len(incompletas)}):** " + ", ".join(incompletas) +
                     ". Sus métricas se calculan sobre las imágenes disponibles; las comparaciones pareadas usan la intersección."))

**Figura 0 — Diseño experimental.** Cobertura de cada celda *modelo × prompt* sobre el conjunto esperado. Una celda gris es una combinación
sin corrida; una celda por debajo de 100 % es una corrida en curso o interrumpida.

In [ ]:
cov = runs_meta.pivot(index="model", columns="prompt", values="cobertura").reindex(index=MODEL_ORDER, columns=PROMPT_ORDER)
fig, ax = plt.subplots(figsize=(1.05 * len(PROMPT_ORDER) + 2.6, 0.55 * len(MODEL_ORDER) + 1.2))
heat(ax, cov.values, [PS[p] for p in PROMPT_ORDER], [ML[m] for m in MODEL_ORDER], fmt="{:.0%}", vmin=0, vmax=1.6)
ax.set_title(f"Cobertura por corrida (n esperado = {N_EXP:,})", loc="left")
fig.tight_layout()
savefig(fig, "00_diseno_experimental")
plt.show()

## 3. Factores verdaderos y líneas base

Los atributos verdaderos se toman **siempre del manifest**. Para el color hay dos verdades posibles:

* **Tono del objeto** $h_i$ (latente de la imagen), discretizado en ocho categorías circulares con $\theta=360h$:
  rojo $[345°,15°)$, naranjo $[15°,40°)$, amarillo $[40°,70°)$, verde $[70°,160°)$, cian $[160°,195°)$, azul $[195°,255°)$,
  morado $[255°,285°)$, rosado $[285°,345°)$. Es la verdad perceptual.
* **Nombre de la referencia** $c_i$ (`text_object_color_name`). Solo es alcanzable reproduciendo el vocabulario de Matplotlib,
  como piden p4 y p4m.

La **línea base mayoritaria** $\max_k \hat\pi_k$ es la exactitud de responder siempre la clase más frecuente; para color con tolerancia
$\pm1$ categoría es $\max_k\sum_{j:\,d(j,k)\le1}\hat\pi_j$.

In [ ]:
COLOR_CATS = ["red", "orange", "yellow", "green", "cyan", "blue", "purple", "pink"]
COLOR_ES = dict(zip(COLOR_CATS, ["rojo", "naranjo", "amarillo", "verde", "cian", "azul", "morado", "rosado"])) | {"achromatic": "acromático"}
HUE_UPPER = [15, 40, 70, 160, 195, 255, 285, 345]
CAT_CENTER = {"red": 0, "orange": 27.5, "yellow": 55, "green": 115, "cyan": 177.5, "blue": 225, "purple": 270, "pink": 315}
CAT_SWATCH = {c: colorsys.hsv_to_rgb(CAT_CENTER[c] / 360, .75, .9) for c in COLOR_CATS}   # aquí el color es el dato


def hue_to_cat(h):
    d = (float(h) * 360) % 360
    for c, u in zip(COLOR_CATS, HUE_UPPER):
        if d < u:
            return c
    return "red"


def cat_distance(a, b):
    if a not in COLOR_CATS or b not in COLOR_CATS:
        return 99
    d = abs(COLOR_CATS.index(a) - COLOR_CATS.index(b))
    return min(d, len(COLOR_CATS) - d)


def circ_deg(a, b):
    # |a - b| circular en grados, con a y b en [0, 1).
    return 360 * np.abs(((np.asarray(a) - np.asarray(b) + 0.5) % 1.0) - 0.5)


truth = man_full.filter(pl.col("id").is_in(expected_ids)).select(
    "id", "image_path", "caption_ref", "object_shape", "object_xpos", "object_ypos", "object_color",
    "background_color", "spotlight_color", pl.col("text_object_color_name").str.to_lowercase().alias("name_true"),
).to_pandas()
truth["cat_true"] = truth["object_color"].map(hue_to_cat)
truth["name_family"] = truth["name_true"].str.extract(r"^(xkcd|tab):")[0].fillna("CSS4")
print(f"{len(truth):,} imágenes · {truth['name_true'].nunique()} nombres de color distintos en la referencia")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), gridspec_kw={"width_ratios": [1.5, 1, 1.6, 0.9]})
ax = axes[0]
vc = truth["object_shape"].value_counts().sort_index()
ax.bar(vc.index.astype(str), vc.values, color=SERIES[0], width=0.6)
ax.set(title="Forma (código)", ylabel="imágenes"); ax.grid(axis="x", visible=False)

ax = axes[1]
G = pd.crosstab(truth["object_ypos"], truth["object_xpos"]).reindex(index=[0, 1, 2], columns=[0, 1, 2]).values
heat(ax, G / G.sum(), ["x=0", "x=1", "x=2"], ["y=0", "y=1", "y=2"], fmt="{:.1%}", vmin=0, vmax=0.25)
ax.set_title("Posición (y, x)")

ax = axes[2]
bins = np.linspace(0, 1, 37); centers = (bins[:-1] + bins[1:]) / 2
counts, _ = np.histogram(truth["object_color"], bins=bins)
ax.bar(centers, counts, width=bins[1] - bins[0], color=[colorsys.hsv_to_rgb(h, .75, .9) for h in centers], edgecolor=SURF, linewidth=1)
for u in HUE_UPPER:
    ax.axvline(u / 360, color=INK3, lw=0.8, ls=":")
ax.set(title="Tono del objeto $h$ y límites de categoría", xlabel="$h\\in[0,1)$", ylabel="imágenes"); ax.grid(axis="x", visible=False)

ax = axes[3]
vc = truth["name_family"].value_counts()
ax.barh(vc.index, vc.values, color=SERIES[0], height=0.55)
ax.set(title="Familia del nombre", xlabel="imágenes"); ax.grid(axis="y", visible=False)
fig.tight_layout()
savefig(fig, "01_factores_verdaderos")
plt.show()

## 4. Extractor de atributos

**Limpieza.** Se aplica `strip_think_tags` (Qwen3.5-Base usa la plantilla del Instruct y a veces emite `</think>` sueltos).

* **Forma.** Léxico de siete clases con sinónimos (`rabbit → hare`, `bust → head`, …). Se registra la primera clase mencionada y el número de
  clases distintas (más de una indica alucinación o copia de un ejemplo).
* **Posición.** Primero pares compuestos (`top-left`, `mid-center`, `upper right`); si no, ejes por separado (`top/upper/bottom/lower`,
  `left/right`); `center/middle` completa el eje no mencionado. Se eliminan giros relacionales (`on top of`, `right side up`).
* **Color.** Menciones en orden de aparición: (i) nombres entre comillas resolubles por `matplotlib.colors` (`"xkcd:bright blue"`),
  (ii) nombres con prefijo sin comillas, (iii) términos del léxico básico y los nombres CSS4 del vocabulario de la referencia.
  Los términos básicos se asignan a su categoría léxica (p. ej. *purple* → morado, aunque el CSS `purple` tenga $\theta=300°$); los demás
  nombres se convierten a HSV y se clasifican por su tono ($S<0.2$ o $V<0.2$ ⇒ acromático). Dos términos contiguos
  (*bluish green*, *yellow-green*) se fusionan tomando el último como núcleo. Cada mención se asigna al **fondo** o a la **luz** si en las
  tres palabras siguientes hay un sustantivo de ese tipo (o, si no, un sustantivo de fondo en las tres anteriores), y al **objeto** en otro caso.
* **p3_json.** Se usa `parse_json_response`; si devuelve un objeto se leen sus campos. Con `STRICT_JSON = True`, una respuesta sin JSON
  recuperable cuenta como **fallo de formato**: sus atributos quedan omitidos (y por tanto errados en la exactitud estricta).

Para cada atributo $a$: $m_{i,a}=\mathbb 1[\hat z_{i,a}\neq\varnothing]$ (mención) y $k_{i,a}=\mathbb 1[\hat z_{i,a}=z_{i,a}]$ (acierto;
omitir es fallar).

In [ ]:
SHAPE_LEXICON = {
    "teapot": ["teapot", "tea pot", "kettle"], "hare": ["hare", "rabbit", "bunny", "bunnies"],
    "dragon": ["dragon"], "cow": ["cow", "bull", "ox", "oxen", "cattle", "calf"], "armadillo": ["armadillo"],
    "horse": ["horse", "pony", "stallion", "mare"], "head": ["head", "bust", "face"],
}


def _term_rx(t):
    return r"\b" + re.escape(t).replace(r"\ ", r"[\s\-]?") + r"(?:s|es)?\b"


SHAPE_RE = {k: re.compile("|".join(_term_rx(t) for t in v), re.I) for k, v in SHAPE_LEXICON.items()}


def find_shapes(text):
    return sorted((m.start(), k) for k, rx in SHAPE_RE.items() for m in rx.finditer(text))


# ── Posición ──
V_MAP = {"top": "top", "upper": "top", "bottom": "bottom", "lower": "bottom",
         "mid": "mid", "middle": "mid", "center": "mid", "centre": "mid"}
H_MAP = {"left": "left", "right": "right", "center": "center", "centre": "center", "middle": "center", "mid": "center"}
PAIR_RE = re.compile(r"\b(top|upper|mid|middle|center|centre|bottom|lower)[\s-]+(left|right|center|centre|middle)\b")
V_RE = re.compile(r"\b(top|upper|bottom|lower|mid)\b")
H_RE = re.compile(r"\b(left|right)\b")
C_RE = re.compile(r"\b(center|centre|middle|centered|centred)\b")
NOISE_RE = re.compile(r"\b(?:on top of|right[\s-]side[\s-]up|right angles?|all right|top[\s-]down)\b")


def parse_position(text):
    t = NOISE_RE.sub(" ", text.lower())
    m = PAIR_RE.search(t)
    if m:
        return H_MAP[m.group(2)], V_MAP[m.group(1)]
    v, h = V_RE.search(t), H_RE.search(t)
    v, h = (V_MAP[v.group(1)] if v else None), (H_MAP[h.group(1)] if h else None)
    if C_RE.search(t):
        v, h = v or "mid", h or "center"
    return h, v


# ── Color ──
COLOR_LEXICON = {
    "red": ["red", "reddish", "crimson", "scarlet", "maroon", "burgundy", "ruby", "cherry"],
    "orange": ["orange", "orangish", "amber", "tangerine", "peach", "rust", "copper", "salmon", "coral"],
    "yellow": ["yellow", "yellowish", "gold", "golden", "mustard", "lemon"],
    "green": ["green", "greenish", "lime", "olive", "chartreuse", "emerald", "mint"],
    "cyan": ["cyan", "teal", "turquoise", "aqua", "aquamarine"],
    "blue": ["blue", "bluish", "navy", "azure", "cobalt", "sapphire", "cerulean"],
    "purple": ["purple", "purplish", "violet", "lavender", "indigo", "lilac", "plum", "mauve"],
    "pink": ["pink", "pinkish", "magenta", "fuchsia", "rose"],
    "achromatic": ["white", "black", "gray", "grey", "silver", "brown", "beige", "tan"],
}
TERM2CAT = {t: c for c, v in COLOR_LEXICON.items() for t in v}
CSS_VOCAB = {n for n in truth["name_true"].unique() if ":" not in n and n not in TERM2CAT}   # p. ej. deepskyblue, springgreen
TERM_RE = re.compile(r"\b(" + "|".join(sorted(map(re.escape, set(TERM2CAT) | CSS_VOCAB), key=len, reverse=True)) + r")\b")
QUOTED_RE = re.compile(r"[\"“”']([^\"“”']{2,40})[\"“”']")
PREFIX_RE = re.compile(r"\b(?:xkcd|tab):\s?[a-z]+(?:[ /][a-z]+){0,3}")
BG_WORDS = {"background", "backdrop", "backgrounds", "wall", "floor", "ground", "surface", "sky"}
LIGHT_WORDS = {"light", "lighting", "lights", "spotlight", "glow", "illumination", "illuminated", "lit", "shadow", "shadows"}
MODIFIERS = {"light", "dark", "bright", "pale", "deep", "vivid", "neon", "pastel", "soft"}
WORD_RE = re.compile(r"[a-z]+(?:['\-][a-z]+)*")


def norm_name(s):
    return " ".join(s.lower().replace("\u00a0", " ").split())


def resolve_color(name):
    # Nombre -> (categoría, tono en [0,1) o None). None si no es un color.
    name = norm_name(name)
    if name in TERM2CAT:
        c = TERM2CAT[name]
        return c, (CAT_CENTER[c] / 360 if c in CAT_CENTER else None)
    if name.startswith("#") or not mcolors.is_color_like(name) or re.fullmatch(r"[\d.\s]+", name):
        return None
    h, s, v = colorsys.rgb_to_hsv(*mcolors.to_rgb(name))
    return ("achromatic", None) if (s < 0.2 or v < 0.2) else (hue_to_cat(h), h)


def color_mentions(text):
    t = text.lower()
    found, taken = [], []
    for m in QUOTED_RE.finditer(t):
        name = norm_name(m.group(1))
        info = resolve_color(name)
        if info:
            found.append({"s": m.start(), "e": m.end(), "name": name, "cat": info[0], "hue": info[1], "quoted": True})
            taken.append((m.start(), m.end()))
    inside = lambda a, b: any(a < e and b > s for s, e in taken)
    for m in PREFIX_RE.finditer(t):
        if inside(m.start(), m.end()):
            continue
        words = m.group(0).replace(": ", ":").split(" ")
        for k in range(len(words), 0, -1):                  # prefijo más largo que sea un color
            info = resolve_color(" ".join(words[:k]))
            if info:
                end = m.start() + len(" ".join(words[:k]))
                found.append({"s": m.start(), "e": end, "name": norm_name(" ".join(words[:k])), "cat": info[0], "hue": info[1], "quoted": False})
                taken.append((m.start(), end))
                break
    for m in TERM_RE.finditer(t):
        if inside(m.start(), m.end()):
            continue
        info = resolve_color(m.group(1))
        found.append({"s": m.start(), "e": m.end(), "name": m.group(1), "cat": info[0], "hue": info[1], "quoted": False})
    found.sort(key=lambda d: d["s"])
    merged = []                                             # "bluish green", "yellow-green": núcleo = último término
    for f in found:
        if merged and not merged[-1]["quoted"] and not f["quoted"] and re.fullmatch(r"[\s\-]*", t[merged[-1]["e"]:f["s"]]):
            f = f | {"s": merged[-1]["s"], "name": t[merged[-1]["s"]:f["e"]]}
            merged[-1] = f
        else:
            merged.append(f)
    for f in merged:
        after = WORD_RE.findall(t[f["e"]:])[:3]
        before = WORD_RE.findall(t[:f["s"]])[-3:]
        if before and before[-1] in MODIFIERS:
            before = before[:-1]
        if any(w in BG_WORDS for w in after):
            f["entity"] = "background"
        elif any(w in LIGHT_WORDS for w in after):
            f["entity"] = "light"
        elif any(w in BG_WORDS for w in before):
            f["entity"] = "background"
        else:
            f["entity"] = "object"
    return merged


def _empty():
    return {"shape": None, "n_shapes": 0, "xpos": None, "ypos": None, "color_name": None, "color_cat": None,
            "color_hue": None, "color_quoted": False, "n_colors": 0, "bg_cat": None}


def _from_text(text):
    out = _empty()
    sh = find_shapes(text)
    out["shape"], out["n_shapes"] = (sh[0][1] if sh else None), len({k for _, k in sh})
    out["xpos"], out["ypos"] = parse_position(text)
    cm = color_mentions(text)
    obj = next((c for c in cm if c["entity"] == "object"), None)
    bg = next((c for c in cm if c["entity"] == "background"), None)
    out["n_colors"] = len(cm)
    out["bg_cat"] = bg["cat"] if bg else None
    if obj:
        out.update(color_name=obj["name"], color_cat=obj["cat"], color_hue=obj["hue"], color_quoted=obj["quoted"])
    return out


def extract(raw, json_mode=False):
    raw = raw or ""
    clean = strip_think_tags(raw)
    base = {"think": "think>" in raw.lower(), "n_words": len(WORD_RE.findall(clean.lower())), "n_chars": len(clean),
            "ends_clean": bool(re.search(r"[.!?)\"'\]}`]\s*$", clean)), "json_ok": None,
            "uses_named": bool(re.search(r"\b(?:xkcd|tab):", clean.lower()))}
    if not json_mode:
        return base | _from_text(clean)
    obj = parse_json_response(clean)
    base["json_ok"] = obj is not None
    if obj is None:
        return base | (_empty() if STRICT_JSON else _from_text(clean))
    f = {k: str(v) for k, v in obj.items() if v is not None}
    out = _empty()
    sh = find_shapes(f.get("shape", ""))
    out["shape"], out["n_shapes"] = (sh[0][1] if sh else None), len({k for _, k in sh})
    out["xpos"] = H_MAP.get(f.get("horizontal", "").strip().lower())
    out["ypos"] = V_MAP.get(f.get("vertical", "").strip().lower())
    col = f.get("color", "").strip()
    info = resolve_color(col.strip("\"'")) if col else None
    cm = color_mentions(col) if col and not info else []
    if info:
        out.update(color_name=norm_name(col.strip("\"'")), color_cat=info[0], color_hue=info[1], n_colors=1)
    elif cm:
        out.update(color_name=cm[-1]["name"], color_cat=cm[-1]["cat"], color_hue=cm[-1]["hue"], n_colors=len(cm))
    return base | out

### 4.1 Validación sobre las referencias

Las referencias tienen contenido conocido: el extractor debe recuperar forma, posición y nombre de color con exactitud $\approx 1$.
Los códigos enteros de forma y posición no traen nombre, así que la biyección código → etiqueta se **infiere** de las referencias y su
*pureza* $\max_k\hat P(\text{etiqueta}=k\mid\text{código})$ mide la precisión del extractor. Además se obtiene la **cota** de concordancia entre
la categoría del nombre de la referencia y la categoría de $h_i$: ni siquiera un modelo que reproduzca exactamente la referencia puede
superarla en la exactitud de categoría de tono.

In [ ]:
ref_ext = pd.DataFrame([extract(t) for t in truth["caption_ref"]], index=truth.index)


def code_map(codes, labels, name):
    ct = pd.crosstab(codes, labels.fillna("∅"))
    valid = ct.drop(columns=[c for c in ("∅",) if c in ct.columns])
    mapping = valid.idxmax(axis=1).to_dict()
    purity = valid.max(axis=1).sum() / ct.values.sum()
    if len(set(mapping.values())) != len(mapping):
        print(f"[aviso] {name}: la asignación no es biyectiva → {mapping}")
    return mapping, purity


SHAPE_NAMES, pur_s = code_map(truth["object_shape"], ref_ext["shape"], "forma")
XPOS_NAMES, pur_x = code_map(truth["object_xpos"], ref_ext["xpos"], "posición x")
YPOS_NAMES, pur_y = code_map(truth["object_ypos"], ref_ext["ypos"], "posición y")
truth["shape_true"] = truth["object_shape"].map(SHAPE_NAMES)
truth["x_true"] = truth["object_xpos"].map(XPOS_NAMES)
truth["y_true"] = truth["object_ypos"].map(YPOS_NAMES)
truth["ref_hue"] = [colorsys.rgb_to_hsv(*mcolors.to_rgb(n))[0] for n in truth["name_true"]]
truth["ref_cat"] = [resolve_color(n)[0] for n in truth["name_true"]]

val = pd.Series({
    "pureza forma": pur_s, "pureza posición x": pur_x, "pureza posición y": pur_y,
    "nombre de color exacto": (ref_ext["color_name"] == truth["name_true"]).mean(),
    "color asignado al objeto": ref_ext["color_name"].notna().mean(),
    "cota: cat(nombre) = cat(h)": (truth["ref_cat"] == truth["cat_true"]).mean(),
    "cota: cat(nombre) a ±1 de cat(h)": np.mean([cat_distance(a, b) <= 1 for a, b in zip(truth["ref_cat"], truth["cat_true"])]),
}, name="valor")
display(val.to_frame().style.format("{:.4f}"))
print("SHAPE_NAMES =", SHAPE_NAMES, "\nXPOS_NAMES  =", XPOS_NAMES, "\nYPOS_NAMES  =", YPOS_NAMES)
for k in ["pureza forma", "pureza posición x", "pureza posición y", "nombre de color exacto"]:
    if val[k] < 0.99:
        print(f"[aviso] {k} = {val[k]:.3f} < 0.99: revisar el extractor antes de interpretar ese atributo.")
bad = truth.loc[ref_ext["color_name"] != truth["name_true"], ["caption_ref", "name_true"]].head(5)
if len(bad):
    display(bad.assign(extraido=ref_ext.loc[bad.index, "color_name"]))
CEIL_CAT = val["cota: cat(nombre) = cat(h)"]
CEIL_ADJ = val["cota: cat(nombre) a ±1 de cat(h)"]

In [ ]:
dlt = ((truth["ref_hue"] - truth["object_color"] + 0.5) % 1.0 - 0.5) * 360
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
axes[0].scatter(truth["object_color"], truth["ref_hue"], s=5, alpha=0.3, color=SERIES[0], linewidths=0)
axes[0].plot([0, 1], [0, 1], color=INK3, lw=1, ls="--")
axes[0].set(xlabel="$h$ del objeto (latente)", ylabel="tono del nombre de la referencia", xlim=(0, 1), ylim=(0, 1),
            title="Tono del latente vs. tono del nombre")
axes[1].hist(dlt, bins=np.arange(-60, 61, 2), color=SERIES[0])
axes[1].axvline(0, color=INK3, lw=1, ls="--")
axes[1].set(xlabel="Δ circular (grados)", ylabel="imágenes",
            title=f"|Δ| mediana {np.median(np.abs(dlt)):.1f}°, P90 {np.percentile(np.abs(dlt), 90):.1f}°")
axes[1].grid(axis="x", visible=False)
fig.tight_layout()
savefig(fig, "02_validacion_tono_referencia")
plt.show()

### 4.2 Extracción sobre las predicciones

Se extraen los atributos de cada predicción (con caché por texto: las respuestas copiadas son idénticas) y se cruzan con la verdad.

In [ ]:
cache = {}


def cached_extract(text, json_mode):
    key = (text, json_mode)
    if key not in cache:
        cache[key] = extract(text, json_mode)
    return cache[key]


# VL-JEPA devuelve captions aunque el prompt pida JSON: el modo JSON solo aplica a los modelos generativos
ext = pd.DataFrame([cached_extract(t, p in JSON_PROMPTS and m not in RETRIEVAL)
                    for t, p, m in zip(pred["prediction"], pred["prompt"], pred["model"])], index=pred.index)
df = pd.concat([pred, ext], axis=1).merge(
    truth[["id", "shape_true", "x_true", "y_true", "object_color", "cat_true", "name_true", "ref_cat", "background_color"]],
    on="id", how="left")
assert df["shape_true"].notna().all(), "predicciones sin verdad en el manifest"

df["shape_m"], df["shape_ok"] = df["shape"].notna(), df["shape"] == df["shape_true"]
df["x_m"], df["x_ok"] = df["xpos"].notna(), df["xpos"] == df["x_true"]
df["y_m"], df["y_ok"] = df["ypos"].notna(), df["ypos"] == df["y_true"]
df["pos_m"], df["pos_ok"] = df["x_m"] & df["y_m"], df["x_ok"] & df["y_ok"]
df["color_m"] = df["color_cat"].notna()
df["color_ok"] = df["color_cat"] == df["cat_true"]
df["color_adj"] = [cat_distance(a, b) <= 1 for a, b in zip(df["color_cat"], df["cat_true"])]
df["name_m"] = df["color_name"].notna()
df["name_ok"] = df["color_name"] == df["name_true"]
df["hue_err"] = np.where(df["color_hue"].notna(), circ_deg(df["color_hue"].astype(float).fillna(0), df["object_color"]), np.nan)
df["macro"] = (df["shape_ok"].astype(int) + df["pos_ok"].astype(int) + df["color_adj"].astype(int)) / 3
df["copy_literal"] = df["copy_object"] = df["copy_any"] = False
print(f"{len(df):,} filas · {len(cache):,} textos distintos extraídos")

## 5. Texto

* **Truncamiento probable:** el texto no termina en puntuación de cierre (típico de alcanzar `max_new_tokens`).
* **Únicas:** $|\{\hat t_{i,r}\}_i|/N_r$. Cerca de 0 indica una respuesta fija (copia o colapso); en prompts de plantilla es esperable que sea bajo.
* **Nombre con prefijo:** fracción que usa `xkcd:`/`tab:`, es decir, que adopta la convención de la referencia.
* **`<think>`:** fracción con etiquetas de razonamiento residuales (se limpian antes de extraer).
* **JSON válido:** solo p3; con `STRICT_JSON` lo que no es JSON cuenta como fallo de formato.

In [ ]:
def q(p):
    f = lambda s: s.quantile(p)
    f.__name__ = f"p{int(100 * p)}"
    return f


g = df.groupby("run", sort=False)
text_tab = pd.DataFrame({
    "palabras_p50": g["n_words"].median(), "palabras_p95": g["n_words"].quantile(0.95),
    "vacias": g["n_words"].apply(lambda s: (s == 0).mean()), "truncadas": 1 - g["ends_clean"].mean(),
    "unicas": g["prediction"].apply(lambda s: s.nunique() / len(s)), "nombre_con_prefijo": g["uses_named"].mean(),
    "color_entre_comillas": g["color_quoted"].mean(), "think": g["think"].mean(),
    "varias_formas": g["n_shapes"].apply(lambda s: (s > 1).mean()),
    "json_valido": g["json_ok"].apply(lambda s: s.dropna().astype(float).mean() if s.notna().any() else np.nan),
}).reindex(RUN_ORDER)
pct = [c for c in text_tab.columns if not c.startswith("palabras")]
display(text_tab.style.format({c: "{:.1%}" for c in pct} | {"palabras_p50": "{:.0f}", "palabras_p95": "{:.0f}"}, na_rep="–"))
save_table(text_tab, "texto_por_corrida")
REF_MED = ref_ext["n_words"].median()
print(f"Referencia: mediana {REF_MED:.0f} palabras")

**Figura 3 — Indicadores de texto por modelo × prompt.** Cada panel es un indicador; las celdas anotan el valor.

In [ ]:
panels = [("palabras_p50", "Palabras (mediana)", "{:.0f}", None), ("truncadas", "Truncadas", "{:.0%}", 1),
          ("unicas", "Respuestas únicas", "{:.0%}", 1), ("color_entre_comillas", "Color entre comillas", "{:.0%}", 1),
          ("think", "Etiquetas <think>", "{:.0%}", 1), ("varias_formas", "Más de una forma", "{:.0%}", 1)]
tt = text_tab.join(runs_meta.set_index("run")[["model", "prompt"]])
fig, axes = plt.subplots(2, 3, figsize=(14.5, 2 * (0.42 * len(MODEL_ORDER) + 1.3)))
for ax, (col, title, fmt, vmax) in zip(axes.flat, panels):
    M = tt.pivot(index="model", columns="prompt", values=col).reindex(index=MODEL_ORDER, columns=PROMPT_ORDER)
    heat(ax, M.values, [PS[p] for p in PROMPT_ORDER], [ML[m] for m in MODEL_ORDER], fmt=fmt,
         vmin=0, vmax=vmax or max(1.0, np.nanmax(M.values) * 1.3), fontsize=7.5)
    ax.set_title(title, loc="left")
fig.tight_layout()
savefig(fig, "03_indicadores_texto")
plt.show()

**Figura 4 — Longitud de las respuestas.** Un panel por prompt; barra gruesa = P25–P75, línea fina = P5–P95, `|` = mediana; escala log.
La línea punteada es la mediana de la referencia.

In [ ]:
fig, axes = plt.subplots(1, len(PROMPT_ORDER), figsize=(2.55 * len(PROMPT_ORDER) + 1.6, 0.42 * len(MODEL_ORDER) + 1.3),
                         sharey=True, squeeze=False)
y = np.arange(len(MODEL_ORDER))
for ax, p in zip(axes[0], PROMPT_ORDER):
    for yi, m in zip(y, MODEL_ORDER):
        s = df.loc[(df["model"] == m) & (df["prompt"] == p), "n_words"].clip(lower=1)
        if s.empty:
            continue
        q05, q25, q50, q75, q95 = s.quantile([.05, .25, .5, .75, .95])
        ax.plot([q05, q95], [yi, yi], color=MODEL_COLORS[m], lw=1.2, alpha=0.7, solid_capstyle="round")
        ax.plot([q25, q75], [yi, yi], color=MODEL_COLORS[m], lw=6, solid_capstyle="butt")
        ax.plot(q50, yi, "|", color=INK, ms=11, mew=2)
    ax.axvline(REF_MED, color=INK2, ls="--", lw=1)
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
    ax.set_title(PS[p]); ax.grid(axis="y", visible=False)
axes[0, 0].set_yticks(y, [ML[m] for m in MODEL_ORDER]); axes[0, 0].invert_yaxis()
fig.supxlabel("palabras por respuesta", fontsize=9.5, color=INK2)
fig.tight_layout()
savefig(fig, "04_longitud")
plt.show()

## 6. Copia de los ejemplos del prompt (p4 vs p4m)

p4 incluye un ejemplo concreto (`A "tab:blue" horse is at the bottom-right…`) y p4m tres distintos. Con `vlmfid.eval.copy_flags` se marcan:

* **copia literal:** la respuesta contiene el color entre comillas seguido del objeto de algún ejemplo;
* **copia de objeto:** nombra el objeto de un ejemplo que **no** es la forma verdadera de la imagen (sin penalizar aciertos reales);
* **copia (cualquiera):** $\mathbb 1[\text{literal}\lor\text{objeto}]$.

La copia de objeto tiene un piso aunque el modelo no copie: nombrar por error el objeto del ejemplo. Ese piso (`copia_objeto_basal`) se
estima con la tasa a la que el mismo modelo nombra erróneamente esos objetos bajo los prompts **sin** ejemplos (p0–p3); solo el exceso sobre
el piso es atribuible al ejemplo. Para VL-JEPA, que recupera captions, ese exceso mide el sesgo de la *consulta* hacia el ejemplo, no una copia.

Una respuesta copiada acierta solo cuando la verdad coincide con el ejemplo, así que la exactitud cruda sobreestima la percepción del modelo
justo donde la copia es alta. Por eso se reporta también la **exactitud sobre lo no copiado** $\hat\alpha_{\neg c}=\sum_i k_i(1-c_i)/\sum_i(1-c_i)$,
con su IC de Wilson, y la exactitud esperada de un **copiador puro** que siempre responde el ejemplo $e$:
$P(s=s_e)$, $P((x,y)=(x_e,y_e))$ y $P(c=c_e)$.

In [ ]:
EX_PROMPTS = [p for p in PROMPT_ORDER if prompt_examples(runs_meta.loc[runs_meta["prompt"] == p, "prompt_text"].iloc[0])]
POS_TOKEN = {f"{v}-{h}": (h, v) for v in ["top", "mid", "bottom"] for h in ["left", "center", "right"]}
copy_rows, ceiling_rows = [], []
for p in EX_PROMPTS:
    text_p = runs_meta.loc[runs_meta["prompt"] == p, "prompt_text"].iloc[0]
    exs = prompt_examples(text_p)
    for e in exs:
        h, v = POS_TOKEN.get(e.position, (None, None))
        ceiling_rows.append({"prompt": PS[p], "ejemplo": e.marker, "P(forma)": (truth["shape_true"] == e.obj).mean(),
                             "P(posición)": ((truth["x_true"] == h) & (truth["y_true"] == v)).mean(),
                             "P(nombre)": (truth["name_true"] == e.color.lower()).mean()})
    idx = df.index[df["prompt"] == p]
    flags = pd.DataFrame([copy_flags(t, exs, s) for t, s in zip(df.loc[idx, "prediction"], df.loc[idx, "shape_true"])], index=idx)
    df.loc[idx, ["copy_literal", "copy_object", "copy_any"]] = flags[["copy_literal", "copy_object", "copy_any"]].values
df["copy_any"] = df["copy_any"].astype(bool)
display(pd.DataFrame(ceiling_rows).style.format({c: "{:.1%}" for c in ["P(forma)", "P(posición)", "P(nombre)"]})
        .set_caption("Exactitud de un copiador puro por ejemplo"))

NO_EX = [p for p in PROMPT_ORDER if p not in EX_PROMPTS]


def basal_object(m, exs):
    # Tasa con que el modelo nombra el objeto de algún ejemplo (sin ser la forma verdadera) bajo los prompts SIN ejemplos:
    # es la 'copia de objeto' que se observaría por puro error de forma, sin copiar nada.
    s = df[(df["model"] == m) & df["prompt"].isin(NO_EX)]
    if s.empty:
        return np.nan
    return np.mean([copy_flags(t, exs, sh)["copy_object"] for t, sh in zip(s["prediction"], s["shape_true"])])


sub = df[df["prompt"].isin(EX_PROMPTS)]
for (run, m, p), gg in sub.groupby(["run", "model", "prompt"], sort=False):
    nc = gg[~gg["copy_any"]]
    exs = prompt_examples(runs_meta.loc[runs_meta["run"] == run, "prompt_text"].iloc[0])
    row = {"run": run, "model": m, "prompt": p, "n": len(gg), "copia_literal": gg["copy_literal"].astype(float).mean(),
           "copia_objeto": gg["copy_object"].astype(float).mean(), "copia_objeto_basal": basal_object(m, exs),
           "copia": gg["copy_any"].mean(),
           "unicas": gg["prediction"].nunique() / len(gg), "n_no_copiadas": len(nc)}
    for a in ["shape", "pos", "color_adj", "name"]:
        col = f"{a}_ok" if a != "color_adj" else a
        row[f"{a}_todas"] = gg[col].mean()
        row[f"{a}_no_copiadas"] = nc[col].mean() if len(nc) else np.nan
        lo, hi = wilson(nc[col].sum(), len(nc))
        row[f"{a}_nc_ic"] = f"[{lo:.1%}, {hi:.1%}]" if len(nc) else "–"
    copy_rows.append(row)
copy_tab = pd.DataFrame(copy_rows).set_index("run")
show = ["copia_literal", "copia_objeto", "copia_objeto_basal", "copia", "unicas", "n_no_copiadas", "shape_todas", "shape_no_copiadas", "shape_nc_ic",
        "pos_todas", "pos_no_copiadas", "pos_nc_ic", "color_adj_todas", "color_adj_no_copiadas"]
display(copy_tab[show].style.format({c: "{:.1%}" for c in show if c not in ("n_no_copiadas",) and not c.endswith("_ic")}
                                    | {"n_no_copiadas": "{:,.0f}"}, na_rep="–"))
save_table(copy_tab.drop(columns=[c for c in copy_tab.columns if c.endswith("_ic")]), "copia_ejemplos")

**Figura 5 — Copia y su efecto.** Izquierda: tasa de copia con p4 (●) y p4m (◆) por modelo. Derecha: exactitud de forma sobre todas las
respuestas (marcador hueco) y sobre las no copiadas (marcador lleno); el segmento une ambas. La línea punteada es la línea base mayoritaria.

In [ ]:
if len(EX_PROMPTS) >= 1:
    MAJ_SHAPE = truth["shape_true"].value_counts(normalize=True).iloc[0]
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 0.5 * len(MODEL_ORDER) + 1.9), sharey=True)
    y = np.arange(len(MODEL_ORDER))
    off = {p: (i - (len(EX_PROMPTS) - 1) / 2) * 0.22 for i, p in enumerate(EX_PROMPTS)}
    pm = {p: mk for p, mk in zip(EX_PROMPTS, ["o", "D", "s", "^"])}
    for yi, m in zip(y, MODEL_ORDER):
        vals = [copy_tab.loc[(copy_tab["model"] == m) & (copy_tab["prompt"] == p), "copia"] for p in EX_PROMPTS]
        xs = [v.iloc[0] if len(v) else np.nan for v in vals]
        if len(EX_PROMPTS) == 2 and np.isfinite(xs).all():
            axes[0].plot(xs, [yi, yi], color=NEUTRAL, lw=2, zorder=1)
        for p, xv in zip(EX_PROMPTS, xs):
            if np.isfinite(xv):
                axes[0].plot(xv, yi, pm[p], color=MODEL_COLORS[m], ms=9, mec=SURF, mew=1.5, zorder=3)
                axes[0].annotate(f"{xv:.0%}", (xv, yi), xytext=(0, 9), textcoords="offset points", ha="center", fontsize=7.5, color=INK2)
        for p in EX_PROMPTS:
            r = copy_tab[(copy_tab["model"] == m) & (copy_tab["prompt"] == p)]
            if r.empty:
                continue
            a, b = r["shape_todas"].iloc[0], r["shape_no_copiadas"].iloc[0]
            yy = yi + off[p]
            if np.isfinite(b):
                axes[1].plot([a, b], [yy, yy], color=MODEL_COLORS[m], lw=1.5, alpha=0.6)
                axes[1].plot(b, yy, pm[p], color=MODEL_COLORS[m], ms=8, mec=SURF, mew=1.2)
            axes[1].plot(a, yy, pm[p], mfc=SURF, mec=MODEL_COLORS[m], mew=1.6, ms=8)
    axes[0].set(xlim=(-0.03, 1.05), title="Tasa de copia de los ejemplos"); axes[0].xaxis.set_major_formatter(PCT)
    axes[1].axvline(MAJ_SHAPE, color=INK3, ls=":", lw=1.2)
    axes[1].set(xlim=(-0.03, 1.03), title="Exactitud de forma: todas (hueco) vs. no copiadas (lleno)"); axes[1].xaxis.set_major_formatter(PCT)
    axes[0].set_yticks(y, [ML[m] for m in MODEL_ORDER]); axes[0].invert_yaxis()
    for ax in axes:
        ax.grid(axis="y", visible=False)
    handles = [plt.Line2D([], [], marker=pm[p], ls="", color=INK2, label=PS[p]) for p in EX_PROMPTS] + \
              [plt.Line2D([], [], color=INK3, ls=":", label="base mayoritaria")]
    fig.tight_layout(rect=(0, 0, 1, 0.9))
    fig.legend(handles=handles, loc="upper center", ncol=len(handles), bbox_to_anchor=(0.5, 0.99))
    savefig(fig, "05_copia_ejemplos")
    plt.show()

## 7. Fidelidad por atributo

Para la corrida $r$ y el atributo $a$, con $N_r$ imágenes:
$$
\hat c_{r,a}=\frac1{N_r}\sum_i m_{i,a},\qquad
\hat\alpha_{r,a}=\frac1{N_r}\sum_i k_{i,a},\qquad
\hat\alpha^{\mid}_{r,a}=\frac{\hat\alpha_{r,a}}{\hat c_{r,a}} .
$$
$\hat\alpha$ penaliza tanto la omisión como el error; $\hat\alpha^{\mid}$ aísla la calidad de lo que sí se dice. Con $N_r=10^4$ la semiamplitud
del IC de Wilson al 95 % es a lo sumo $z\sqrt{0.25/10^4}\approx0.98$ pp, de modo que diferencias de pocos puntos entre corridas ya son
distinguibles del ruido de muestreo (no así del sesgo del extractor, acotado en §4.1).

Atributos: **forma**, **x**, **y**, **posición** conjunta $(x,y)$, **color** (categoría de $h$ exacta y a $\pm1$), **nombre** exacto de la referencia
(p. ej. `xkcd:bright yellow green`; si la referencia usa un nombre CSS como `blue`, la palabra sin comillas también cuenta) e **índice macro** $\tfrac13\big(k_{\text{forma}}+k_{\text{pos}}+k_{\text{color}\pm1}\big)$.

In [ ]:
ATTRS = {"shape": ("shape_m", "shape_ok", "Forma"), "x": ("x_m", "x_ok", "Posición x"), "y": ("y_m", "y_ok", "Posición y"),
         "pos": ("pos_m", "pos_ok", "Posición (x,y)"), "color": ("color_m", "color_ok", "Color (categoría)"),
         "color_adj": ("color_m", "color_adj", "Color ±1"), "name": ("name_m", "name_ok", "Nombre exacto")}
ATTR_LABELS = {k: v[2] for k, v in ATTRS.items()} | {"macro": "Índice macro"}

base = truth
MAJ = {"shape": base["shape_true"].value_counts(normalize=True).iloc[0],
       "x": base["x_true"].value_counts(normalize=True).iloc[0], "y": base["y_true"].value_counts(normalize=True).iloc[0],
       "pos": (base["x_true"] + base["y_true"]).value_counts(normalize=True).iloc[0],
       "color": base["cat_true"].value_counts(normalize=True).iloc[0],
       "name": base["name_true"].value_counts(normalize=True).iloc[0]}
vc = base["cat_true"].value_counts(normalize=True)
MAJ["color_adj"] = max(sum(vc.get(c2, 0) for c2 in COLOR_CATS if cat_distance(c1, c2) <= 1) for c1 in COLOR_CATS)

rows = []
for (run, m, p), gg in df.groupby(["run", "model", "prompt"], sort=False):
    n = len(gg)
    for a, (mc, kc, _) in ATTRS.items():
        k, mm = int(gg[kc].sum()), int(gg[mc].sum())
        lo, hi = wilson(k, n)
        rows.append({"run": run, "model": m, "prompt": p, "attr": a, "n": n, "mencion": mm / n, "exactitud": k / n,
                     "ic_lo": float(lo), "ic_hi": float(hi), "exactitud_cond": k / mm if mm else np.nan})
acc = pd.DataFrame(rows)
macro = df.groupby("run", sort=False)["macro"].mean()

acc_wide = acc.pivot(index="run", columns="attr", values="exactitud").reindex(index=RUN_ORDER, columns=list(ATTRS))
acc_wide["macro"] = macro.reindex(RUN_ORDER)
acc_wide.columns = [ATTR_LABELS[c] for c in acc_wide.columns]
display(acc_wide.style.format("{:.1%}").background_gradient(cmap=SEQ, vmin=0, vmax=1, axis=None)
        .set_caption("Exactitud estricta por corrida (omitir = error)"))
display(pd.Series({ATTR_LABELS[k]: v for k, v in MAJ.items()}, name="base mayoritaria").to_frame().T.style.format("{:.1%}"))
save_table(acc_wide, "exactitud_por_corrida")
acc.to_csv(TAB_DIR / "fidelidad_largo.csv", index=False)

**Figura 6 — Exactitud por modelo × prompt.** Un panel por atributo, misma escala en todos. El subtítulo indica la línea base mayoritaria
(y, para el color, la cota de §4.1). Es la vista principal para comparar prompts dentro de un modelo y modelos dentro de un prompt.

In [ ]:
panels = ["shape", "pos", "x", "y", "color", "color_adj", "name", "macro"]
fig, axes = plt.subplots(2, 4, figsize=(17, 2 * (0.42 * len(MODEL_ORDER) + 1.45)))
for ax, a in zip(axes.flat, panels):
    if a == "macro":
        s = pd.DataFrame({"v": macro}).join(runs_meta.set_index("run")[["model", "prompt"]])
    else:
        s = acc[acc["attr"] == a].rename(columns={"exactitud": "v"})
    M = s.pivot(index="model", columns="prompt", values="v").reindex(index=MODEL_ORDER, columns=PROMPT_ORDER)
    heat(ax, M.values, [PS[p] for p in PROMPT_ORDER], [ML[m] for m in MODEL_ORDER], fontsize=7.5)
    sub = f"base {MAJ[a]:.0%}" if a in MAJ else ""
    if a == "color":
        sub += f" · cota {CEIL_CAT:.0%}"
    if a == "color_adj":
        sub += f" · cota {CEIL_ADJ:.0%}"
    ax.set_title(f"{ATTR_LABELS[a]}" + (f"\n{sub}" if sub else ""), loc="left", fontsize=10)
for ax in axes[:, 1:].flat:
    ax.set_yticklabels([])
fig.tight_layout()
savefig(fig, "06_exactitud_modelo_prompt")
plt.show()

**Figura 7 — Descomposición del resultado.** Para cada corrida, fracción de imágenes con el atributo *correcto*, *incorrecto* (mencionado pero
errado) u *omitido*. Distingue a los modelos que se equivocan de los que simplemente no lo dicen; lo omitido bajo p0 no es evidencia de que
el modelo no perciba el atributo.

In [ ]:
OUT_COLS = {"correcto": SERIES[0], "incorrecto": SERIES[1], "omitido": NEUTRAL}
dec = [("shape", "Forma"), ("pos", "Posición (x,y)"), ("color_adj", "Color ±1"), ("name", "Nombre exacto")]
H = 0.27 * len(RUN_ORDER) + 1.6
fig, axes = plt.subplots(1, len(dec), figsize=(3.3 * len(dec) + 2.4, H), sharey=True)
y = np.arange(len(RUN_ORDER))
for ax, (a, title) in zip(axes, dec):
    s = acc[acc["attr"] == a].set_index("run").reindex(RUN_ORDER)
    parts = {"correcto": s["exactitud"].values, "incorrecto": (s["mencion"] - s["exactitud"]).values, "omitido": 1 - s["mencion"].values}
    left = np.zeros(len(RUN_ORDER))
    for k, v in parts.items():
        ax.barh(y, v, left=left, height=0.78, color=OUT_COLS[k], edgecolor=SURF, linewidth=1.2, label=k)
        left += v
    for b in np.flatnonzero(runs_meta["model"].values[1:] != runs_meta["model"].values[:-1]) + 0.5:
        ax.axhline(b, color=INK3, lw=0.6)
    ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(PCT); ax.set_title(title); ax.grid(axis="y", visible=False)
axes[0].set_yticks(y, RUN_ORDER, fontsize=7.5); axes[0].invert_yaxis()
hdl, lbl = axes[0].get_legend_handles_labels()
fig.tight_layout(rect=(0, 0, 1, 1 - 0.55 / H))
fig.legend(hdl, lbl, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.0))
savefig(fig, "07_descomposicion_resultado")
plt.show()

## 8. Sensibilidad al prompt

**Figura 8.** Exactitud por prompt, una línea por modelo (prompts ordenados de menor a mayor especificación). Una pendiente pronunciada indica
que el resultado depende más de *cómo se pregunta* que de *lo que el modelo percibe*. El contraste p0 → p3 separa percepción de formato;
p4/p4m además de formato incluye ejemplos que pueden copiarse (§6).

In [ ]:
sens = ["shape", "pos", "color_adj", "macro"]
fig, axes = plt.subplots(1, len(sens), figsize=(3.7 * len(sens) + 0.8, 3.9), sharey=True)
x = np.arange(len(PROMPT_ORDER))
for ax, a in zip(axes, sens):
    for m in MODEL_ORDER:
        rr = runs_meta[runs_meta["model"] == m]
        vals = [macro.get(r, np.nan) if a == "macro" else
                acc.loc[(acc["run"] == r) & (acc["attr"] == a), "exactitud"].squeeze() for r in rr["run"]]
        xs = [PROMPT_ORDER.index(p) for p in rr["prompt"]]
        ls = "--" if m in RETRIEVAL else "-"
        ax.plot(xs, vals, marker=MODEL_MARKERS[m], color=MODEL_COLORS[m], ls=ls, ms=7, mec=SURF, mew=1)
    if a in MAJ:
        ax.axhline(MAJ[a], color=INK3, ls=":", lw=1.1)
    ax.set_xticks(x, [PS[p] for p in PROMPT_ORDER]); ax.set_ylim(0, 1); ax.yaxis.set_major_formatter(PCT)
    ax.set_title(ATTR_LABELS[a])
handles = [plt.Line2D([], [], color=MODEL_COLORS[m], marker=MODEL_MARKERS[m], ls="--" if m in RETRIEVAL else "-",
                      label=ML[m] + (" (recuperación)" if m in RETRIEVAL else "")) for m in MODEL_ORDER] + \
          [plt.Line2D([], [], color=INK3, ls=":", label="base mayoritaria")]
fig.tight_layout(rect=(0, 0, 1, 0.86))
fig.legend(handles=handles, loc="upper center", ncol=min(len(handles), 6), bbox_to_anchor=(0.5, 0.99))
savefig(fig, "08_sensibilidad_prompt")
plt.show()

# Robustez vs. techo por modelo (sobre el índice macro)
mm = pd.DataFrame({"macro": macro}).join(runs_meta.set_index("run")[["model", "prompt"]])
rob = mm.groupby("model").agg(macro_media=("macro", "mean"), macro_max=("macro", "max"), macro_sd=("macro", "std"),
                              n_prompts=("macro", "size"))
rob["mejor_prompt"] = mm.loc[mm.groupby("model")["macro"].idxmax()].set_index("model")["prompt"].map(PS)
rob = rob.reindex(MODEL_ORDER).rename(index=ML)
display(rob.style.format({"macro_media": "{:.1%}", "macro_max": "{:.1%}", "macro_sd": "{:.3f}"}))
save_table(rob, "robustez_por_modelo")

**Figura 9 — Cobertura vs. exactitud condicional.** Cada punto es una corrida (forma = modelo, etiqueta = prompt); las curvas grises son
exactitud estricta constante $\hat\alpha=\hat c\,\hat\alpha^{\mid}$. Arriba a la izquierda: dice poco pero bien; abajo a la derecha: lo dice todo
pero se equivoca. Marcador hueco = recuperación (VL-JEPA).

In [ ]:
cov_attrs = ["shape", "pos", "color_adj"]
fig, axes = plt.subplots(1, len(cov_attrs), figsize=(4.3 * len(cov_attrs), 4.3), sharey=True)
cc = np.linspace(0.01, 1, 200)
for ax, a in zip(axes, cov_attrs):
    for lvl in (0.25, 0.5, 0.75):
        ax.plot(cc[cc >= lvl], lvl / cc[cc >= lvl], color=GRID, lw=1.2, zorder=0)
        ax.text(lvl + 0.01, 0.985, f"α={lvl:.2f}", fontsize=7, color=INK3, va="top")
    for _, r in acc[acc["attr"] == a].iterrows():
        ret = r["model"] in RETRIEVAL
        c = MODEL_COLORS[r["model"]]
        ax.scatter(r["mencion"], r["exactitud_cond"], s=58, marker=MODEL_MARKERS[r["model"]],
                   facecolor=SURF if ret else c, edgecolor=c if ret else SURF, linewidth=1.6 if ret else 1.2, zorder=3)
        ax.annotate(PS[r["prompt"]], (r["mencion"], r["exactitud_cond"]), xytext=(4, 3), textcoords="offset points",
                    fontsize=6.5, color=INK2)
    lo_c = acc.loc[acc["attr"] == a, "mencion"].min()
    ax.set(xlim=(max(-0.02, lo_c - 0.1), 1.06), ylim=(-0.02, 1.02), xlabel="cobertura $\\hat c$", title=ATTR_LABELS[a])
    ax.xaxis.set_major_formatter(PCT); ax.yaxis.set_major_formatter(PCT)
axes[0].set_ylabel("exactitud condicional $\\hat\\alpha^{\\mid}$")
handles = [plt.Line2D([], [], marker=MODEL_MARKERS[m], ls="", color=MODEL_COLORS[m], mfc=SURF if m in RETRIEVAL else MODEL_COLORS[m],
                      label=ML[m]) for m in MODEL_ORDER]
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.legend(handles=handles, loc="upper center", ncol=len(handles), bbox_to_anchor=(0.5, 0.99))
savefig(fig, "09_cobertura_vs_condicional")
plt.show()

## 9. Control Qwen3.5 Base vs Instruct

Mismo tamaño, misma plantilla de entrada; solo cambia el ajuste por instrucciones. Por prompt y atributo se compara la exactitud **sobre las
mismas imágenes** (intersección) con la prueba exacta de McNemar: con $b$ = aciertos solo de Instruct y $c$ = aciertos solo de Base, bajo
$H_0$ $b\sim\mathrm{Bin}(b+c,\tfrac12)$. Los $p$-valores se ajustan por Holm sobre todas las comparaciones de la tabla.

In [ ]:
def paired(df_, run_a, run_b, col):
    A = df_.loc[df_["run"] == run_a].set_index("id")[col]
    B = df_.loc[df_["run"] == run_b].set_index("id")[col]
    ids = A.index.intersection(B.index)
    a, b = A.loc[ids].astype(bool).values, B.loc[ids].astype(bool).values
    return len(ids), a.mean(), b.mean(), int((~a & b).sum()), int((a & ~b).sum())


QB, QI = "qwen35_08b_base", "qwen35_08b_instruct"
bi_rows = []
if {QB, QI} <= set(MODEL_ORDER):
    for p in PROMPT_ORDER:
        ra = runs_meta.loc[(runs_meta["model"] == QB) & (runs_meta["prompt"] == p), "run"]
        rb = runs_meta.loc[(runs_meta["model"] == QI) & (runs_meta["prompt"] == p), "run"]
        if ra.empty or rb.empty:
            continue
        for a in ["shape", "pos", "color_adj", "name"]:
            col = ATTRS[a][1]
            n, pa, pb, b_, c_ = paired(df, ra.iloc[0], rb.iloc[0], col)
            bi_rows.append({"prompt": PS[p], "atributo": ATTR_LABELS[a], "n_pareado": n, "Base": pa, "Instruct": pb,
                            "Δ (I−B)": pb - pa, "b (solo I)": b_, "c (solo B)": c_, "p": mcnemar_exact(b_, c_)})
if bi_rows:
    bi = pd.DataFrame(bi_rows)
    bi["p_holm"] = holm(bi["p"])
    bi["sig"] = np.where(bi["p_holm"] < ALPHA, "✓", "")
    display(bi.style.format({"Base": "{:.1%}", "Instruct": "{:.1%}", "Δ (I−B)": "{:+.1%}", "p": "{:.2g}", "p_holm": "{:.2g}",
                             "n_pareado": "{:,}"}).background_gradient(cmap=DIV, subset=["Δ (I−B)"], vmin=-0.5, vmax=0.5))
    save_table(bi, "qwen_base_vs_instruct", index=False)

    atts = bi["atributo"].unique()
    fig, axes = plt.subplots(1, len(atts), figsize=(3.4 * len(atts) + 0.6, 0.45 * bi["prompt"].nunique() + 1.7), sharey=True)
    for ax, att in zip(np.atleast_1d(axes), atts):
        s = bi[bi["atributo"] == att].reset_index(drop=True)
        yy = np.arange(len(s))
        for i, r in s.iterrows():
            ax.plot([r["Base"], r["Instruct"]], [i, i], color=NEUTRAL, lw=2, zorder=1)
        ax.plot(s["Base"], yy, MODEL_MARKERS[QB], color=MODEL_COLORS[QB], ms=8, mec=SURF, label=ML[QB], zorder=3)
        ax.plot(s["Instruct"], yy, MODEL_MARKERS[QI], color=MODEL_COLORS[QI], ms=8, mec=SURF, label=ML[QI], zorder=3)
        for i, r in s.iterrows():
            if r["sig"]:
                ax.annotate("*", (max(r["Base"], r["Instruct"]), i), xytext=(6, -3), textcoords="offset points", color=INK, fontsize=11)
        ax.set_yticks(yy, s["prompt"]); ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(PCT)
        ax.set_title(att); ax.grid(axis="y", visible=False)
    np.atleast_1d(axes)[0].invert_yaxis()
    h, l = np.atleast_1d(axes)[0].get_legend_handles_labels()
    fig.tight_layout(rect=(0, 0, 1, 0.86))
    fig.legend(h + [plt.Line2D([], [], ls="", marker="$*$", color=INK, label=f"p Holm < {ALPHA}")],
               l + [f"p Holm < {ALPHA}"], loc="upper center", ncol=3, bbox_to_anchor=(0.5, 0.99))
    savefig(fig, "10_qwen_base_vs_instruct")
    plt.show()
else:
    print("Falta alguna corrida de Qwen3.5 Base o Instruct.")

## 10. Errores con el mejor prompt de cada modelo

Para cada modelo se toma el prompt de mayor índice macro (tabla de §8) y se examina **qué** confunde.

In [ ]:
BEST = mm.loc[mm.groupby("model")["macro"].idxmax()].set_index("model")["prompt"].reindex(MODEL_ORDER).to_dict()
best_run = {m: runs_meta.loc[(runs_meta["model"] == m) & (runs_meta["prompt"] == p), "run"].iloc[0] for m, p in BEST.items()}
print({ML[m]: PS[p] for m, p in BEST.items()})
SHAPES = [SHAPE_NAMES[k] for k in sorted(SHAPE_NAMES)]

fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(3.3 * len(MODEL_ORDER), 3.7), sharey=True, squeeze=False)
for ax, m in zip(axes[0], MODEL_ORDER):
    s = df[df["run"] == best_run[m]]
    ct = pd.crosstab(s["shape_true"], s["shape"].fillna("∅")).reindex(index=SHAPES, columns=SHAPES + ["∅"], fill_value=0)
    heat(ax, ct.div(ct.sum(axis=1), axis=0).values, SHAPES + ["∅"], SHAPES, fmt="{:.0%}", fontsize=6.5)
    ax.tick_params(axis="x", rotation=60)
    ax.set_title(f"{ML[m]} · {PS[BEST[m]]}", fontsize=9.5)
axes[0, 0].set_ylabel("forma verdadera")
fig.supxlabel("forma predicha (∅ = omitida)", fontsize=9.5, color=INK2)
fig.suptitle("Confusión de forma (normalizada por fila)", x=0.01, ha="left", fontweight="semibold")
fig.tight_layout()
savefig(fig, "11_confusion_forma")
plt.show()

In [ ]:
XS, YS = ["left", "center", "right"], ["top", "mid", "bottom"]
fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(2.75 * len(MODEL_ORDER), 2.8), squeeze=False)
for ax, m in zip(axes[0], MODEL_ORDER):
    s = df[df["run"] == best_run[m]]
    G = s.pivot_table(index="y_true", columns="x_true", values="pos_ok", aggfunc="mean").reindex(index=YS, columns=XS)
    heat(ax, G.values, XS, YS, fontsize=8)
    ax.set_title(f"{ML[m]} · {PS[BEST[m]]}", fontsize=9.5)
fig.suptitle("Exactitud de posición (x,y) según la celda verdadera", x=0.01, ha="left", fontweight="semibold")
fig.tight_layout()
savefig(fig, "12_posicion_grilla")
plt.show()

**Figura 13 — Color según el tono verdadero.** Arriba: exactitud de categoría (±1) por intervalo de $h$ para el mejor prompt de cada modelo;
las líneas verticales son los límites de categoría, donde es esperable una caída (un tono cerca del límite admite dos nombres razonables).
Abajo: matriz de confusión de categoría para el modelo de mayor exactitud de color.

In [ ]:
bins = np.linspace(0, 1, 25); centers = (bins[:-1] + bins[1:]) / 2
fig = plt.figure(figsize=(13, 8.2))
gs = fig.add_gridspec(2, 1, height_ratios=[1, 1.15], hspace=0.45)
ax = fig.add_subplot(gs[0])
for m in MODEL_ORDER:
    s = df[df["run"] == best_run[m]]
    b = np.clip(np.digitize(s["object_color"], bins) - 1, 0, len(centers) - 1)
    accb = s.groupby(b)["color_adj"].mean().reindex(range(len(centers)))
    ax.plot(centers, accb.values, marker=MODEL_MARKERS[m], color=MODEL_COLORS[m], ms=6, mec=SURF, label=f"{ML[m]} · {PS[BEST[m]]}")
for u in HUE_UPPER:
    ax.axvline(u / 360, color=INK3, lw=0.8, ls=":")
lims = [0] + [u / 360 for u in HUE_UPPER[:-1]] + [HUE_UPPER[-1] / 360]
for c, lo, hi in zip(COLOR_CATS[1:], lims[1:-1], lims[2:]):
    ax.text((lo + hi) / 2, 1.04, COLOR_ES[c], ha="center", fontsize=7.5, color=INK2, transform=ax.get_xaxis_transform())
ax.scatter(centers, np.full_like(centers, -0.035), c=[colorsys.hsv_to_rgb(h, .75, .9) for h in centers], s=38, marker="s",
           transform=ax.get_xaxis_transform(), clip_on=False)
ax.tick_params(axis="x", pad=16)
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="tono verdadero $h$", ylabel="exactitud color ±1")
ax.yaxis.set_major_formatter(PCT)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))

m_best = max(MODEL_ORDER, key=lambda m: df.loc[df["run"] == best_run[m], "color_ok"].mean())
s = df[df["run"] == best_run[m_best]]
cols_ = COLOR_CATS + ["achromatic", "∅"]
ct = pd.crosstab(s["cat_true"], s["color_cat"].fillna("∅")).reindex(index=COLOR_CATS, columns=cols_, fill_value=0)
ax2 = fig.add_subplot(gs[1])
heat(ax2, ct.div(ct.sum(axis=1), axis=0).values, [COLOR_ES.get(c, c) for c in cols_], [COLOR_ES[c] for c in COLOR_CATS], fontsize=7.5)
ax2.set(xlabel="categoría predicha", ylabel="categoría de $h$")
ax2.set_title(f"Confusión de color — {ML[m_best]} · {PS[BEST[m_best]]}", loc="left")
savefig(fig, "13_color_vs_tono")
plt.show()

herr = df[df["color_hue"].notna()].groupby("run", sort=False)["hue_err"].describe(percentiles=[.5, .9])[["count", "50%", "90%"]]
display(herr.reindex([r for r in RUN_ORDER if r in herr.index]).rename(columns={"count": "n con tono", "50%": "|Δθ| mediana (°)",
        "90%": "|Δθ| P90 (°)"}).style.format({"n con tono": "{:,.0f}", "|Δθ| mediana (°)": "{:.1f}", "|Δθ| P90 (°)": "{:.1f}"}))

**Figura 14 — Color del objeto vs. color del fondo.** Un error frecuente en escenas con fondo saturado es nombrar el color del fondo como si
fuera el del objeto. Se compara, cuando el color predicho del objeto es errado, si coincide con la categoría del fondo.

In [ ]:
truth_bg = df["background_color"].map(hue_to_cat)
rows = []
for run, gg in df.groupby("run", sort=False):
    wrong = gg[gg["color_m"] & ~gg["color_ok"]]
    if len(wrong) == 0:
        continue
    bg_hit = (wrong["color_cat"] == truth_bg.loc[wrong.index]).mean()
    chance = np.mean([(truth_bg.loc[wrong.index] == c).mean() * (wrong["color_cat"] == c).mean() for c in COLOR_CATS]) * len(COLOR_CATS)
    rows.append({"run": run, "errores_color": len(wrong), "igual_al_fondo": bg_hit, "esperado_por_azar": chance})
bgt = pd.DataFrame(rows).set_index("run").reindex([r for r in RUN_ORDER if r in {x["run"] for x in rows}])
display(bgt.style.format({"errores_color": "{:,}", "igual_al_fondo": "{:.1%}", "esperado_por_azar": "{:.1%}"}))
save_table(bgt, "contaminacion_fondo")

## 11. Comparaciones pareadas entre modelos y dificultad por imagen

Cada modelo con su mejor prompt, sobre las mismas imágenes: McNemar exacto para forma, posición y color ±1, con Holm sobre la familia completa.

In [ ]:
pair_rows = []
for i, m1 in enumerate(MODEL_ORDER):
    for m2 in MODEL_ORDER[i + 1:]:
        for a in ["shape", "pos", "color_adj"]:
            n, p1, p2, b_, c_ = paired(df, best_run[m1], best_run[m2], ATTRS[a][1])
            pair_rows.append({"A": f"{ML[m1]} · {PS[BEST[m1]]}", "B": f"{ML[m2]} · {PS[BEST[m2]]}", "atributo": ATTR_LABELS[a],
                              "n": n, "acc_A": p1, "acc_B": p2, "Δ (B−A)": p2 - p1, "p": mcnemar_exact(b_, c_)})
pairs = pd.DataFrame(pair_rows)
if len(pairs):
    pairs["p_holm"] = holm(pairs["p"])
    display(pairs.style.format({"acc_A": "{:.1%}", "acc_B": "{:.1%}", "Δ (B−A)": "{:+.1%}", "p": "{:.2g}", "p_holm": "{:.2g}", "n": "{:,}"})
            .background_gradient(cmap=DIV, subset=["Δ (B−A)"], vmin=-0.5, vmax=0.5))
    save_table(pairs, "mcnemar_modelos", index=False)

**Figura 15 — Dificultad por imagen.** Para cada imagen, fracción de las corridas *generativas* que aciertan la forma. Una masa en 0 son
imágenes que ningún modelo resuelve con ningún prompt (candidatas a inspección visual); la tabla desglosa por forma verdadera.

In [ ]:
gen = df[~df["model"].isin(RETRIEVAL)]
diff = gen.groupby("id").agg(frac_forma=("shape_ok", "mean"), frac_pos=("pos_ok", "mean"), frac_color=("color_adj", "mean"),
                             shape_true=("shape_true", "first"))
fig, axes = plt.subplots(1, 3, figsize=(13, 3.3), sharey=True)
for ax, col, t in zip(axes, ["frac_forma", "frac_pos", "frac_color"], ["Forma", "Posición (x,y)", "Color ±1"]):
    ax.hist(diff[col], bins=np.linspace(0, 1, 21), color=SERIES[0], edgecolor=SURF)
    ax.set(title=t, xlabel="fracción de corridas generativas que aciertan"); ax.xaxis.set_major_formatter(PCT)
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("imágenes")
fig.tight_layout()
savefig(fig, "15_dificultad_por_imagen")
plt.show()
dtab = diff.groupby("shape_true")[["frac_forma", "frac_pos", "frac_color"]].mean().sort_values("frac_forma")
dtab["nunca_forma"] = diff.groupby("shape_true")["frac_forma"].apply(lambda s: (s == 0).mean())
display(dtab.style.format("{:.1%}"))
save_table(dtab, "dificultad_por_forma")

## 12. Ejemplos cualitativos

Imágenes elegidas al azar (semilla fija) entre las que tienen predicción en todas las corridas. Se muestra la imagen si `data/raw` está
disponible, la referencia y la respuesta de cada modelo para dos prompts contrastantes.

In [ ]:
N_SHOW = 4
SHOW_PROMPTS = [p for p in ["p1_attributes", "p4m_multiexample"] if p in PROMPT_ORDER] or PROMPT_ORDER[:2]
common = set(truth["id"])
for r in RUN_ORDER:
    common &= set(df.loc[df["run"] == r, "id"])
pick = list(RNG.choice(sorted(common), size=min(N_SHOW, len(common)), replace=False)) if common else []
root = data_root(VARIANT)
tpath = truth.set_index("id")
if pick and (root / tpath.at[pick[0], "image_path"]).exists():
    from PIL import Image
    fig, axes = plt.subplots(1, len(pick), figsize=(3 * len(pick), 3.2))
    for ax, i in zip(np.atleast_1d(axes), pick):
        ax.imshow(Image.open(root / tpath.at[i, "image_path"]))
        ax.set_title(f"{i}\n{tpath.at[i, 'shape_true']} · {tpath.at[i, 'y_true']}-{tpath.at[i, 'x_true']} · {COLOR_ES[tpath.at[i, 'cat_true']]}",
                     fontsize=8)
        ax.axis("off")
    fig.tight_layout()
    plt.show()
else:
    print(f"(imágenes no disponibles en {root}; se muestran solo los textos)")
idx = df.set_index(["id", "run"])
for i in pick:
    display(Markdown(f"**{i}** — referencia: *{tpath.at[i, 'caption_ref']}*  \n"
                     f"verdad: {tpath.at[i, 'shape_true']}, {tpath.at[i, 'y_true']}-{tpath.at[i, 'x_true']}, "
                     f"tono {COLOR_ES[tpath.at[i, 'cat_true']]} (`{tpath.at[i, 'name_true']}`)"))
    rows = []
    for m in MODEL_ORDER:
        for p in SHOW_PROMPTS:
            r = f"{ML[m]} · {PS[p]}"
            if (i, r) in idx.index:
                x_ = idx.loc[(i, r)]
                rows.append({"modelo": ML[m], "prompt": PS[p], "respuesta": x_["prediction"][:220],
                             "forma": "✓" if x_["shape_ok"] else "✗", "pos": "✓" if x_["pos_ok"] else "✗",
                             "color±1": "✓" if x_["color_adj"] else "✗", "copia": "sí" if x_["copy_any"] else ""})
    display(pd.DataFrame(rows).set_index(["modelo", "prompt"]))

## 13. Exportación y limitaciones

In [ ]:
keep_cols = ["exp_id", "run", "model", "prompt", "id", "shape", "shape_true", "shape_ok", "xpos", "ypos", "x_true", "y_true", "pos_ok",
             "color_name", "color_cat", "cat_true", "name_true", "color_ok", "color_adj", "name_ok", "hue_err", "copy_any",
             "n_words", "ends_clean", "think", "json_ok", "macro"]
out = df[keep_cols].copy()
out["json_ok"] = out["json_ok"].astype("boolean")
out.to_parquet(TAB_DIR / "atributos_extraidos.parquet", index=False)
resumen = acc.pivot_table(index=["model", "prompt"], columns="attr", values="exactitud").reindex(columns=list(ATTRS))
resumen["macro"] = mm.set_index(["model", "prompt"])["macro"]
resumen = resumen.rename(index=ML, level=0).rename(index=PS, level=1).rename(columns=ATTR_LABELS)
save_table(resumen, "tabla_informe")
print("Tablas:", sorted(p.name for p in TAB_DIR.iterdir()))
print("Figuras:", sorted(p.name for p in FIG_DIR.glob("*.png")))

**Limitaciones.**

1. **Extractor léxico.** Está validado sobre las referencias (§4.1), cuyo lenguaje es regular; en texto libre (p0) puede fallar con sinónimos
   o paráfrasis no previstas. Una omisión del extractor se contabiliza como omisión del modelo, así que la *mención* en p0 es una cota inferior.
2. **Color.** La categoría de tono es una discretización arbitraria con fronteras duras; por eso se reporta también la tolerancia $\pm1$ y
   el error angular. La exactitud de nombre exacto solo es alcanzable con p4/p4m (vocabulario de Matplotlib), y aun así la cota perceptual
   está limitada por la concordancia nombre–tono de la propia referencia (§4.1).
3. **Asignación de color a entidades.** La heurística de ventana (±3 palabras) puede atribuir al objeto un color del fondo en frases largas;
   la tabla de §10 cuantifica cuánto coinciden los errores con el color del fondo.
4. **VL-JEPA.** Sus salidas son captions recuperados de *train*: su fidelidad refleja la calidad de la recuperación, no de la generación, y
   sus indicadores de texto (formato, comillas, longitud) no son comparables.
5. **Copia.** La detección se basa en el texto de los ejemplos del prompt; una copia parcial parafraseada (p. ej. otro color con el objeto del
   ejemplo que coincide con la verdad) no se marca, por diseño, para no penalizar aciertos.
6. **Inferencia.** Las pruebas de McNemar condicionan en el conjunto de imágenes (variabilidad de muestreo de imágenes), no en la
   aleatoriedad de decodificación: todas las corridas usan decodificación voraz y un único *seed*.